# PIR Route Agent — Installation Windows 11 + VS Code

Ce notebook constitue le **guide d'installation de la V1** du projet.

Objectif :
1. ouvrir le repository GitHub déjà cloné dans VS Code ;
2. créer un environnement Python isolé ;
3. installer les bibliothèques nécessaires à l'analyse GPX ;
4. installer et tester Ollama ;
5. télécharger et tester un LLM local ;
6. vérifier que Python peut appeler le LLM via l'API locale ;
7. préparer la machine pour le développement des agents.

> Ce notebook est pensé pour **Windows 11** et **VS Code**.
>
> Il ne recrée pas le notebook d'Ilya. On s'inspire uniquement de son idée utile :
> **faire tourner un LLM localement et l'appeler depuis Python**.

## 0. Architecture visée

```text
GPX
 │
 ▼
Parsing + rééchantillonnage spatial
 │
 ▼
Agent 1 : choix de l'échelle / du détecteur
 │
 ├── pente
 ├── montée / descente
 ├── lacets
 ├── sinuosité
 └── autres métriques géométriques
 │
 ▼
Annotations atomiques
 │
 ▼
Agent 2 : fusion déterministe en portions
 │
 ▼
Portions + ensemble d'annotations
 │
 ▼
Agent 3 : LLM de verbalisation
 │
 ▼
Description naturelle
```

Pour la V1, on se limite volontairement à des caractéristiques calculables depuis un GPX :
- forte pente ;
- pente douce ;
- montée ;
- descente ;
- lacets ;
- sinuosité.

Les caractéristiques comme forêt, ombre, panorama ou proximité d'un cours d'eau viendront plus tard avec des sources externes.

## 1. Pré-requis

Le repository GitHub doit déjà être cloné sur la machine.

Dans PowerShell :

```powershell
cd C:\chemin\vers\votre-repository
git status
```

Résultat attendu :

```text
On branch main
Your branch is up to date with 'origin/main'.
```

## 2. Ouvrir le repository dans VS Code

Depuis PowerShell :

```powershell
code .
```

Si `code` n'est pas disponible :
1. ouvrir VS Code ;
2. **File → Open Folder** ;
3. sélectionner le repository.

Extensions VS Code recommandées :
- **Python** — Microsoft
- **Pylance** — Microsoft
- **Jupyter** — Microsoft

Le projet principal sera développé en fichiers `.py`.
Le notebook sert ici à documenter et valider l'installation.

## 3. Installer Miniforge / Conda

Vérifier :

```powershell
conda --version
```

Si Conda existe déjà, passer à l'étape suivante.

Sinon :
1. installer **Miniforge3 Windows x86_64** ;
2. ouvrir **Miniforge Prompt** ;
3. vérifier `conda --version` ;
4. initialiser PowerShell :

```powershell
conda init powershell
```

5. fermer puis rouvrir PowerShell.

## 4. Créer l'environnement Python

```powershell
conda create -n pir-route python=3.11 -y
conda activate pir-route
python --version
```

On vise **Python 3.11.x**.

Le terminal doit afficher :

```text
(pir-route) PS C:\...
```

## 5. Installer les dépendances Python de la V1

```powershell
python -m pip install --upgrade pip
python -m pip install numpy pandas gpxpy shapely pyproj scipy pydantic requests matplotlib pytest
```

| Package | Rôle |
|---|---|
| `gpxpy` | lecture et parsing GPX |
| `numpy` | calcul numérique |
| `pandas` | tableaux et annotations |
| `shapely` | géométrie |
| `pyproj` | projections / coordonnées |
| `scipy` | filtrage / calcul scientifique |
| `pydantic` | structures de données fiables |
| `requests` | appels HTTP vers Ollama |
| `matplotlib` | visualisation / debug |
| `pytest` | tests automatiques |

In [ ]:
import sys
print(sys.version)
print(sys.executable)

In [ ]:
import numpy
import pandas
import gpxpy
import shapely
import pyproj
import scipy
import pydantic
import requests
import matplotlib

print("ENVIRONNEMENT PYTHON PIR OK")

## 6. Sélectionner l'interpréteur dans VS Code

Dans VS Code :

```text
Ctrl + Shift + P
```

puis :

```text
Python: Select Interpreter
```

Choisir l'interpréteur contenant :

```text
pir-route
Python 3.11.x
```

Ensuite ouvrir :

```text
Terminal → New Terminal
```

Si nécessaire :

```powershell
conda activate pir-route
```

## 7. Test rapide de l'environnement

Créer `test_environment.py` :

```python
import sys
import numpy
import pandas
import gpxpy
import shapely
import pyproj
import scipy
import pydantic
import requests

print("Python :", sys.version)
print("Executable :", sys.executable)
print("Toutes les dépendances sont disponibles.")
```

Puis :

```powershell
python test_environment.py
```

Le chemin Python affiché doit contenir l'environnement `pir-route`.

## 8. Installer Ollama sur Windows 11

Installer Ollama pour Windows, puis fermer et rouvrir PowerShell.

Vérifier :

```powershell
ollama --version
```

Le serveur local sera disponible sur :

```text
http://localhost:11434
```

## 9. Télécharger le modèle local

Modèle initial recommandé :

```powershell
ollama pull llama3.1:8b
```

Vérifier :

```powershell
ollama list
```

Si la machine est trop limitée, utiliser plus tard un modèle 3B/4B plus léger et modifier la constante `MODEL` dans le code.

## 10. Tester le modèle directement

```powershell
ollama run llama3.1:8b
```

Prompt de test :

```text
Réponds uniquement par un JSON valide.

Nous analysons un itinéraire de randonnée.
Quelle fenêtre choisir parmi [25, 50, 100, 150, 250] mètres
pour rechercher une série de lacets ?

Format :
{"window_m": nombre}
```

Le but est uniquement de vérifier :

```text
Terminal
  ↓
Ollama
  ↓
Llama
  ↓
réponse structurée
```

Quitter avec `/bye`.

## 11. Tester l'API locale Ollama depuis Python

Architecture :

```text
Python
  ↓ HTTP
localhost:11434
  ↓
Ollama
  ↓
LLM
```

In [ ]:
import requests

URL = "http://localhost:11434/api/chat"

payload = {
    "model": "llama3.1:8b",
    "messages": [
        {
            "role": "user",
            "content": '''
Réponds uniquement avec un JSON valide.

Pour détecter une série de lacets sur un itinéraire,
choisis une fenêtre parmi :
25, 50, 100, 150, 250 mètres.

Format :
{"window_m": 150}
'''
        }
    ],
    "stream": False,
    "options": {
        "temperature": 0
    }
}

response = requests.post(URL, json=payload, timeout=120)
response.raise_for_status()

print(response.json()["message"]["content"])

## 12. Jalon de validation

Avant de développer l'architecture :

```text
[ ] environnement Conda `pir-route`
[ ] imports Python OK
[ ] `ollama run llama3.1:8b` fonctionne
[ ] Python peut appeler http://localhost:11434/api/chat
```

Quand ces quatre cases sont validées, la machine est prête.

## 13. Structure cible du repository

```text
project/
│
├── data/
│   ├── input/
│   └── output/
│
├── src/
│   ├── __init__.py
│   ├── gpx_loader.py
│   ├── resampling.py
│   │
│   ├── detectors/
│   │   ├── __init__.py
│   │   ├── slope.py
│   │   ├── ascent.py
│   │   ├── descent.py
│   │   ├── switchbacks.py
│   │   └── sinuosity.py
│   │
│   ├── models.py
│   ├── ollama_client.py
│   ├── agent1.py
│   ├── agent2.py
│   └── agent3.py
│
├── tests/
├── main.py
├── ENVIRONMENT.md
├── CONTEXT.md
├── requirements.txt
├── environment.yml
└── README.md
```

## 14. Principe scientifique à respecter

Le LLM ne calcule pas lui-même les métriques géométriques.

```text
LLM
→ choisit un outil / une fenêtre

Python
→ calcule réellement

LLM
→ interprète / verbalise
```

Cela améliore la reproductibilité et permet de comparer scientifiquement différentes granularités.